# Demo Churn ML — do dado bruto à decisão

> Talk-track: "Vamos percorrer o ciclo completo de um projeto de churn: dados, features, modelo, registro, previsões e monitoramento — tudo dentro do Lakehouse, com o Unity Catalog governando cada etapa."

Unidade de predição: **um cliente em uma data de referência**. Alvo: **cancela nos próximos 30 dias**. Decisão suportada: **priorização do time de retenção**.

## 0. Setup — semente, catálogo e estilo

> Talk-track: "Tudo reproduzível: semente fixa, schema único `workspace.recife_dug` e nomes totalmente qualificados."

In [0]:
# Setup global: semente fixa, namespace, linha do tempo e estilo de gráfico para tema escuro.
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42  # semente fixa -> demo ao vivo idempotente (dá para re-executar o notebook inteiro com segurança)
random.seed(SEED)
np.random.seed(SEED)

CATALOG = "workspace"
SCHEMA = "recife_dug"
FQN = f"{CATALOG}.{SCHEMA}"          # sempre nomes de tabela totalmente qualificados
MODEL_UC = f"{FQN}.modelo_churn"
EXPERIMENTO = "/Users/lucas.almeida8@gmail.com/churn_recife_dug"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQN}")
try:
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA}")
    print("USE CATALOG / USE SCHEMA: OK")
except Exception as e:
    # Peculiaridade conhecida desta sessão serverless: nomes em duas partes
    # podem gerar erro de namespace aninhado. Inofensivo — todas as instruções
    # abaixo usam nomes totalmente qualificados (workspace.recife_dug.*).
    print(f"USE indisponível ({str(e)[:60]}...); usando nomes totalmente qualificados.")

# Linha do tempo fixa do mundo sintético (ano 2026)
DATA_INI_EVENTOS = pd.Timestamp("2025-12-01")   # cobre a janela de 30 dias da primeira data de referência
DATA_FIM_EVENTOS = pd.Timestamp("2026-06-30")   # comportamento observável até 30 de junho
DATAS_REFERENCIA = pd.date_range("2026-01-05", "2026-06-01", freq="W-MON")  # 22 segundas-feiras, inclusive
REF_SCORING = pd.Timestamp("2026-06-01")

# Padrões do matplotlib para tema escuro (slides)
plt.rcParams.update({
    "figure.facecolor": "#1c1c1c", "axes.facecolor": "#1c1c1c",
    "savefig.facecolor": "#1c1c1c", "text.color": "#f0f0f0",
    "axes.labelcolor": "#f0f0f0", "axes.edgecolor": "#9e9e9e",
    "xtick.color": "#f0f0f0", "ytick.color": "#f0f0f0",
    "grid.color": "#4a4a4a", "grid.alpha": 0.55, "axes.grid": True,
    "font.size": 11, "figure.figsize": (10, 4.5), "figure.dpi": 100,
})
AZUL, VERDE, VERMELHO, AMARELO = "#4fc3f7", "#81c784", "#ef5350", "#ffd54f"

print(f"Datas de referência: {len(DATAS_REFERENCIA)} segundas-feiras, "
      f"de {DATAS_REFERENCIA[0].date()} a {DATAS_REFERENCIA[-1].date()}.")

### Por que este notebook NÃO precisa de GPU

> Talk-track: "Pergunta que sempre vem: preciso de GPU? Não."

* Este é um modelo **tabular** com apenas 3 features numéricas. Regressão logística e gradient boosting treinam em **CPU em segundos** — a GPU não aceleraria nada e custaria muito mais.
* GPU vale a pena quando há **operações massivamente paralelas de matrizes**: deep learning (CNNs, transformers), **LLMs grandes**, fine-tuning de modelos de linguagem, ou visão computacional em escala.
* Aqui usamos **serverless compute**: sobe em segundos, escala sozinho, cobra por segundo usado — o tamanho certo para tabular.

## 1. Dados sintéticos — um SaaS fictício

> Talk-track: "Geramos dados realistas de 5 mil clientes — uso, reclamações, pagamentos e assinaturas — com problemas de qualidade injetados de propósito, porque dados reais nunca vêm limpos."

Churn correlaciona com: **poucas sessões, muitas reclamações, muitos dias sem pagamento confirmado**.

In [0]:
# === 1.1 População sintética: ~5.000 clientes ============================
# Uma propensão latente de churn dirige TODOS os comportamentos (poucas
# sessões, muitas reclamações, pagamentos instáveis) -> o modelo precisa
# APRENDER a correlação.
rng = np.random.default_rng(SEED)

N = 5000
IDS_NOMEADOS = [101, 102, 103]  # Ana, Bruno, Carlos - construídos à mão adiante
IDS = np.array([i for i in range(1, N + 1) if i not in IDS_NOMEADOS])

p = rng.beta(2, 5, size=len(IDS))  # propensão latente de churn em [0, 1]

cli = pd.DataFrame({
    "cliente_id": IDS,
    "data_inicio": rng.choice(pd.date_range("2025-01-01", "2025-11-30"), size=len(IDS)),
})
cli["plano"] = rng.choice(["mensal", "trimestral", "anual"], size=len(IDS), p=[0.7, 0.2, 0.1])
cli["mensalidade"] = rng.choice([29.9, 49.9, 79.9], size=len(IDS))

# Parâmetros de comportamento derivados da propensão latente
cli["sessoes_por_semana"] = (1 - p) * 9 + 0.3    # arriscado -> poucas sessões
cli["reclamacoes_por_mes"] = p * 3.5            # arriscado -> muitas reclamações
cli["prob_pagamento_confirmado"] = 1 - 0.45 * p  # arriscado -> pagamentos falhos/atrasados (leve: o mês final de quem cancela é o extremo)

# Cancelamento: quem e quando (todas as datas <= 2026-06-30, logo os
# rótulos são verificáveis até 2026-07-01)
cancela = rng.random(len(IDS)) < (p * 0.95)
cli["data_cancelamento"] = pd.to_datetime(np.where(
    cancela,
    rng.choice(pd.date_range("2026-01-02", "2026-06-30"), size=len(IDS)),
    pd.NaT,
))

print(f"{len(cli)} clientes genéricos | taxa de cancelamento: "
      f"{cli['data_cancelamento'].notna().mean():.1%}")
display(cli.head(5))

In [0]:
# === 1.2 Eventos de uso, reclamações, pagamentos (todos correlacionados com churn) ===
ev_rows, rec_rows, pag_rows = [], [], []
CATEGORIAS = ["atendimento", "cobranca", "produto", "performance"]

for cli_row in cli.itertuples(index=False):
    inicio = cli_row.data_inicio
    cancel = cli_row.data_cancelamento if pd.notna(cli_row.data_cancelamento) else None
    fim = min(cancel or DATA_FIM_EVENTOS, DATA_FIM_EVENTOS)
    if fim <= inicio:
        continue
    # o comportamento normal vai até 30 dias antes do cancelamento (se houver)
    fim_normal = min(fim, cancel - pd.Timedelta(days=30)) if cancel else fim
    if fim_normal < inicio:
        fim_normal = inicio
    semanas = max((fim_normal - inicio).days / 7.0, 0.5)

    # ---- sessões de uso, fase normal (tipo_evento = 'sessao') ----
    n_sess = int(rng.poisson(cli_row.sessoes_por_semana * semanas))
    for d in rng.integers(0, (fim_normal - inicio).days + 1, size=n_sess):
        ev_rows.append((cli_row.cliente_id, inicio + pd.Timedelta(days=int(d)), "sessao"))

    # ---- reclamações, fase normal ----
    n_rec = int(rng.poisson(cli_row.reclamacoes_por_mes * semanas / 4.3))
    for d in rng.integers(0, (fim_normal - inicio).days + 1, size=n_rec):
        rec_rows.append((cli_row.cliente_id, inicio + pd.Timedelta(days=int(d)),
                         str(rng.choice(CATEGORIAS))))

    # ---- pagamentos mensais, fase normal: 'confirmado' x 'tentativa' ----
    dia_venc = int(rng.integers(5, 29))
    for mes_ini in pd.date_range(inicio, fim_normal, freq="MS"):
        venc = mes_ini + pd.Timedelta(days=min(dia_venc, 27))
        if inicio <= venc <= fim_normal:
            if rng.random() < cli_row.prob_pagamento_confirmado:
                pag_rows.append((cli_row.cliente_id, venc, "confirmado", float(cli_row.mensalidade)))
            elif rng.random() < 0.7:
                pag_rows.append((cli_row.cliente_id, venc, "tentativa", float(cli_row.mensalidade)))

    # ---- DETERIORAÇÃO: os 30 dias ANTES do cancelamento são bem diferentes.
    # É isso que torna as features preditivas no ponto no tempo: o uso quase
    # para, as reclamações disparam e o último pagamento costuma falhar ('tentativa').
    if cancel is not None and (cancel - inicio).days > 30:
        d_ini = max(inicio, cancel - pd.Timedelta(days=30))
        dias_det = max((cancel - d_ini).days, 1)
        n_sess_d = int(rng.poisson(0.4))                      # quase para de usar
        for d in rng.integers(0, dias_det, size=n_sess_d):
            ev_rows.append((cli_row.cliente_id, d_ini + pd.Timedelta(days=int(d)), "sessao"))
        n_rec_d = int(rng.poisson(3.5))                       # reclamações disparam
        for d in rng.integers(0, dias_det, size=n_rec_d):
            rec_rows.append((cli_row.cliente_id, d_ini + pd.Timedelta(days=int(d)), "cobranca"))
        if rng.random() < 0.85:                                # pagamento final falha
            venc = cancel - pd.Timedelta(days=int(rng.integers(5, 29)))
            if venc > d_ini:
                pag_rows.append((cli_row.cliente_id, venc, "tentativa", float(cli_row.mensalidade)))

eventos = pd.DataFrame(ev_rows, columns=["cliente_id", "data_evento", "tipo_evento"])
reclamacoes = pd.DataFrame(rec_rows, columns=["cliente_id", "data_reclamacao", "categoria"])
pagamentos = pd.DataFrame(pag_rows, columns=["cliente_id", "data_pagamento", "status", "valor"])

# === Clientes construídos à mão (devem bater EXATAMENTE com os slides) ==================
# Ana (101) - risco ALTO, cancela em 2026-06-18 -> rótulo 'sim' na ref. 2026-06-01
# perfil no dia da previsão: 1 sessão (7d), 3 reclamações (30d), 24 dias sem pagamento confirmado
ana_sessoes = ["2026-01-15", "2026-03-20", "2026-04-10", "2026-05-12", "2026-05-28"]  # só 1 sessão na janela de 7 dias (deterioração)
ana_reclamacoes = ["2026-02-10", "2026-05-05", "2026-05-12", "2026-05-20", "2026-06-10"]  # 06-10 = dado do FUTURO (demo de vazamento)
ana_pagamentos = [("2026-02-20", "confirmado"), ("2026-03-20", "confirmado"), ("2026-04-20", "confirmado"),
                  ("2026-05-08", "confirmado"), ("2026-05-15", "tentativa"), ("2026-05-28", "tentativa")]
# último confirmado em 08/05 -> 24 dias; os pagamentos de maio FALHAM (deterioração, como na população)

# Bruno (102) - risco BAIXO, permanece (rótulo 'não'): 18 sessões na janela de 7 dias
_dias = pd.date_range("2026-05-26", "2026-06-01")
bruno_sessoes = ([d for d in _dias[:4] for _ in range(3)] + [d for d in _dias[4:] for _ in range(2)]
                + list(pd.date_range("2026-01-02", "2026-05-25", freq="3D")))
bruno_pagamentos = [("2026-01-30", "confirmado"), ("2026-02-27", "confirmado"), ("2026-03-30", "confirmado"),
                    ("2026-04-30", "confirmado"), ("2026-05-30", "confirmado")]  # 2 dias antes da referência

# Carlos (103) - risco MÉDIO, permanece: perfil intermediário entre Bruno e o arquétipo de churner
_carlos_sess = ["2026-01-20", "2026-02-15", "2026-03-18", "2026-04-12", "2026-05-04", "2026-05-30", "2026-06-01"]  # 2 sessões na janela de 7 dias
carlos_reclamacoes = ["2026-01-18", "2026-05-25", "2026-05-29"]  # 2 reclamações na janela de 30 dias
carlos_pagamentos = [("2026-03-10", "confirmado"), ("2026-04-10", "confirmado"),
                     ("2026-05-10", "confirmado"), ("2026-05-20", "tentativa")]

def _sess(cid, datas): return [(cid, pd.Timestamp(d), "sessao") for d in datas]
def _rec(cid, datas):  return [(cid, pd.Timestamp(d), "cobranca") for d in datas]
def _pag(cid, lista):  return [(cid, pd.Timestamp(d), s, 49.9) for d, s in lista]

eventos = pd.concat([eventos, pd.DataFrame(
    _sess(101, ana_sessoes) + _sess(102, bruno_sessoes) + _sess(103, _carlos_sess),
    columns=eventos.columns)], ignore_index=True)
reclamacoes = pd.concat([reclamacoes, pd.DataFrame(
    _rec(101, ana_reclamacoes) + _rec(103, carlos_reclamacoes),
    columns=reclamacoes.columns)], ignore_index=True)
pagamentos = pd.concat([pagamentos, pd.DataFrame(
    _pag(101, ana_pagamentos) + _pag(102, bruno_pagamentos) + _pag(103, carlos_pagamentos),
    columns=pagamentos.columns)], ignore_index=True)

clientes = pd.DataFrame({"cliente_id": [101, 102, 103], "nome": ["Ana", "Bruno", "Carlos"]})

assinaturas = cli[["cliente_id", "data_inicio", "plano", "mensalidade", "data_cancelamento"]].copy()
assinaturas = pd.concat([assinaturas, pd.DataFrame([
    {"cliente_id": 101, "data_inicio": pd.Timestamp("2025-03-15"), "plano": "mensal",
     "mensalidade": 49.9, "data_cancelamento": pd.Timestamp("2026-06-18")},
    {"cliente_id": 102, "data_inicio": pd.Timestamp("2024-08-10"), "plano": "mensal",
     "mensalidade": 49.9, "data_cancelamento": pd.NaT},
    {"cliente_id": 103, "data_inicio": pd.Timestamp("2025-01-20"), "plano": "trimestral",
     "mensalidade": 79.9, "data_cancelamento": pd.NaT},
])], ignore_index=True)

# === Problemas de qualidade INJETADOS DE PROPÓSITO ==================================
eventos.insert(0, "id_evento", np.arange(len(eventos), dtype="int64"))  # id único do evento (detecção de duplicados)
# (a) 150 clientes sem NENHUM registro de uso (assinatura ativa, zero eventos)
sem_uso = rng.choice(IDS, size=150, replace=False)
eventos = eventos[~eventos.cliente_id.isin(sem_uso)]
# (b) 100 clientes sem NENHUM registro de pagamento
sem_pag = rng.choice(IDS, size=100, replace=False)
pagamentos = pagamentos[~pagamentos.cliente_id.isin(sem_pag)]
# (c) eventos duplicados: cópias exatas de 3% do volume de eventos (mesmo id_evento)
duplicados = eventos.sample(frac=0.03, random_state=SEED)
eventos = pd.concat([eventos, duplicados], ignore_index=True)
# (d) algumas reclamações datadas ANTES do início da assinatura (datas inconsistentes)
ini_por_cliente = assinaturas.set_index("cliente_id")["data_inicio"]
alvo = reclamacoes.sample(20, random_state=SEED).index
reclamacoes.loc[alvo, "data_reclamacao"] = (reclamacoes.loc[alvo, "cliente_id"].map(ini_por_cliente)
                                            - pd.Timedelta(days=5))

print(f"eventos: {len(eventos):,} (duplicatas injetadas: {len(duplicados):,}) | "
      f"reclamações: {len(reclamacoes):,} | pagamentos: {len(pagamentos):,} | assinaturas: {len(assinaturas):,}")

In [0]:
# === 1.3 Gravar como tabelas no Unity Catalog (idempotente: overwrite) ============
def salvar(pdf, tabela, comentario):
    destino = f"{FQN}.{tabela}"
    (spark.createDataFrame(pdf)
          .write.mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(destino))
    # escapa aspas simples para o texto em português nunca quebrar o literal SQL
    spark.sql(f"COMMENT ON TABLE {destino} IS '{comentario.replace(chr(39), chr(39)*2)}'")
    return destino

salvar(assinaturas, "assinaturas", "Assinaturas: início e cancelamento por cliente (dados sintéticos do demo)")
salvar(clientes, "clientes", "Cadastro reduzido de clientes; nome conhecido apenas para os 3 clientes do demo")
salvar(eventos, "eventos_uso", "Eventos de uso do produto (tipo sessao); duplicidades injetadas de propósito")
salvar(reclamacoes, "reclamacoes", "Reclamações de clientes por data e categoria")
salvar(pagamentos, "pagamentos", "Pagamentos mensais: status 'confirmado' ou 'tentativa'")

# Comentários de colunas - governança: metadados em português no Unity Catalog
comentarios_colunas = [
    ("assinaturas", "cliente_id", "Identificador único do cliente"),
    ("assinaturas", "data_inicio", "Data de início da assinatura"),
    ("assinaturas", "data_cancelamento", "Data de cancelamento; NULL enquanto a assinatura está ativa"),
    ("eventos_uso", "id_evento", "Identificador único do evento (usado para detectar duplicidades)"),
    ("eventos_uso", "data_evento", "Data em que a sessão ocorreu"),
    ("eventos_uso", "tipo_evento", "Tipo do evento (atualmente sessao)"),
    ("reclamacoes", "data_reclamacao", "Data de abertura da reclamação"),
    ("reclamacoes", "categoria", "Categoria da reclamação (atendimento, cobranca, produto, performance)"),
    ("pagamentos", "data_pagamento", "Data do pagamento"),
    ("pagamentos", "status", "confirmado (quitado) ou tentativa (falhou)"),
    ("clientes", "nome", "Nome do cliente, quando conhecido"),
]
for tabela, coluna, texto in comentarios_colunas:
    spark.sql(f"COMMENT ON COLUMN {FQN}.{tabela}.{coluna} IS '{texto}'")

resumo = pd.DataFrame({
    "tabela": ["clientes", "assinaturas", "eventos_uso", "reclamacoes", "pagamentos"],
    "linhas": [len(clientes), len(assinaturas), len(eventos), len(reclamacoes), len(pagamentos)],
}).sort_values("tabela")
display(resumo)

### Os três clientes do slide (data de referência = 2026-06-01)

| Cliente | Sessões (7d) | Reclamações (30d) | Dias sem pagto confirmado | Situação |
| --- | --- | --- | --- | --- |
| **Ana** (101) | 1 (28/05) | 3 (última em 20/05) | 24 (último confirmado em 08/05; pagamentos de maio falham) | Cancela em **18/06** → rótulo **sim** |
| **Bruno** (102) | 18 | 0 | 2 (último em 30/05) | Continua cliente → rótulo **não** |
| **Carlos** (103) | 2 (30/05 e 01/06) | 2 (25/05 e 29/05) | 22 (último em 10/05) | Continua cliente → perfil **médio** |

A 4ª reclamação da Ana foi registrada em **10/06** — dado **futuro** em relação à data da previsão. Ela existe na tabela, mas **não pode** entrar como feature (seção 4).

## 2. Conhecer os dados — qualidade antes de modelo

> Talk-track: "Antes de qualquer modelo, perguntamos: em quem confiamos? Quem cancelou e quando? Onde estão os problemas?"

In [0]:
# === 2.1 Verificações de qualidade: uma pergunta por fonte ========================
def scalar(sql):
    return spark.sql(sql).first()[0]

total_eventos = scalar(f"SELECT COUNT(*) FROM {FQN}.eventos_uso")
eventos_unicos = scalar(f"SELECT COUNT(DISTINCT id_evento) FROM {FQN}.eventos_uso")
dupl = total_eventos - eventos_unicos

sem_uso = scalar(f"""
  SELECT COUNT(*) FROM {FQN}.assinaturas a
  WHERE NOT EXISTS (SELECT 1 FROM {FQN}.eventos_uso e WHERE e.cliente_id = a.cliente_id)""")

sem_pag = scalar(f"""
  SELECT COUNT(*) FROM {FQN}.assinaturas a
  WHERE NOT EXISTS (SELECT 1 FROM {FQN}.pagamentos g WHERE g.cliente_id = a.cliente_id)""")

rec_inconsistentes = scalar(f"""
  SELECT COUNT(*) FROM {FQN}.reclamacoes c
  JOIN {FQN}.assinaturas a ON a.cliente_id = c.cliente_id
  WHERE to_date(c.data_reclamacao) < to_date(a.data_inicio)""")

canc = spark.sql(f"""SELECT COUNT(*) n, MIN(to_date(data_cancelamento)) ini,
                        MAX(to_date(data_cancelamento)) fim
                 FROM {FQN}.assinaturas WHERE data_cancelamento IS NOT NULL""").first()

def exemplar(tabela, cols):
    r = spark.sql(f"SELECT {cols} FROM {FQN}.{tabela} LIMIT 1").first()
    return " | ".join(str(x) for x in r)

checks = pd.DataFrame([
    {"Fonte": "eventos_uso", "Exemplo de registro": exemplar("eventos_uso", "id_evento, cliente_id, to_date(data_evento), tipo_evento"),
     "Pergunta": "Há eventos duplicados?", "Resultado": f"Sim: {dupl:,} linhas duplicadas ({dupl/total_eventos:.1%} do total)"},
    {"Fonte": "eventos_uso", "Exemplo de registro": exemplar("eventos_uso", "id_evento, cliente_id, to_date(data_evento), tipo_evento"),
     "Pergunta": "Há clientes ativos sem nenhum registro de uso?", "Resultado": f"Sim: {sem_uso} clientes"},
    {"Fonte": "pagamentos", "Exemplo de registro": exemplar("pagamentos", "cliente_id, to_date(data_pagamento), status, valor"),
     "Pergunta": "Há clientes sem nenhum pagamento registrado?", "Resultado": f"Sim: {sem_pag} clientes"},
    {"Fonte": "reclamacoes", "Exemplo de registro": exemplar("reclamacoes", "cliente_id, to_date(data_reclamacao), categoria"),
     "Pergunta": "Há datas inconsistentes (reclamação antes do início da assinatura)?", "Resultado": f"Sim: {rec_inconsistentes} reclamações"},
    {"Fonte": "assinaturas", "Exemplo de registro": exemplar("assinaturas", "cliente_id, to_date(data_inicio), plano, to_date(data_cancelamento)"),
     "Pergunta": "Conseguimos identificar quem cancelou e quando?", "Resultado": f"Sim: {canc['n']:,} cancelamentos entre {canc['ini']} e {canc['fim']}"},
])
display(checks)

In [0]:
# === 2.2 Limpeza: decisões explícitas e documentadas =============================
# DECISÃO 1 (duplicados): eventos com a mesma id_evento são a MESMA sessão
# registrada duas vezes -> manter uma única ocorrência.
(spark.table(f"{FQN}.eventos_uso")
     .dropDuplicates(["id_evento"])
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{FQN}.eventos_uso_limpo"))

# DECISÃO 2 (datas inconsistentes): reclamação registrada ANTES do início da
# assinatura é erro de cadastro -> descartar, em acordo com o negócio.
(spark.sql(f"""
  SELECT c.cliente_id, c.data_reclamacao, c.categoria
  FROM {FQN}.reclamacoes c
  JOIN {FQN}.assinaturas a ON a.cliente_id = c.cliente_id
  WHERE to_date(c.data_reclamacao) >= to_date(a.data_inicio)""")
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{FQN}.reclamacoes_limpo"))

# DECISÃO 3 (pagamentos): status 'tentativa' é um fato real e PERMANECE na
# tabela; a feature 'dias_desde_ultimo_pagamento' usará apenas 'confirmado'.

spark.sql(f"COMMENT ON TABLE {FQN}.eventos_uso_limpo IS 'Eventos de uso deduplicados por id_evento (fonte: eventos_uso)'")
spark.sql(f"COMMENT ON TABLE {FQN}.reclamacoes_limpo IS 'Reclamações com datas consistentes com o início da assinatura (fonte: reclamacoes)'")

antes = pd.DataFrame({"etapa": ["eventos (bruto)", "eventos (limpo)", "reclamações (bruto)", "reclamações (limpo)"],
                      "linhas": [total_eventos,
                                 scalar(f"SELECT COUNT(*) FROM {FQN}.eventos_uso_limpo"),
                                 scalar(f"SELECT COUNT(*) FROM {FQN}.reclamacoes"),
                                 scalar(f"SELECT COUNT(*) FROM {FQN}.reclamacoes_limpo")]})
display(antes)
print("Duplicados removidos:", dupl, "| Reclamações inconsistentes descartadas:", rec_inconsistentes)

## 3. Engenharia de features — correta no ponto no tempo

> Talk-track: "A pergunta que guia tudo: QUE informação estava disponível NA data da previsão? Só o que existia até a data_referencia pode virar feature."

Para cada **(cliente, data_referencia)** — todas as segundas-feiras de 05/01 a 01/06/2026 — calculamos **apenas com dados de data <= data_referencia**:

* `sessoes_ultimos_7_dias` — janela inclusiva: data_referencia−6 até data_referencia
* `reclamacoes_ultimos_30_dias` — janela inclusiva: data_referencia−29 até data_referencia
* `dias_desde_ultimo_pagamento` — dias desde o último pagamento **confirmado**

In [0]:
# === 3.1 Cálculo das features no ponto no tempo (Spark SQL) ======================
# População: clientes ATIVOS em cada data de referência.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW populacao_ref AS
SELECT a.cliente_id, r.data_referencia
FROM {FQN}.assinaturas a
CROSS JOIN (
  SELECT explode(sequence(to_date('2026-01-05'), to_date('2026-06-01'), interval 1 week)) AS data_referencia
) r
WHERE to_date(a.data_inicio) <= r.data_referencia
  AND (a.data_cancelamento IS NULL OR to_date(a.data_cancelamento) > r.data_referencia)""")

# Features: APENAS dados com data <= data_referencia entram em cada janela.
df_feat = spark.sql(f"""
WITH sess AS (
  SELECT p.cliente_id, p.data_referencia, COUNT(DISTINCT e.id_evento) AS sessoes_ultimos_7_dias
  FROM populacao_ref p
  JOIN {FQN}.eventos_uso_limpo e
    ON e.cliente_id = p.cliente_id
   AND to_date(e.data_evento) BETWEEN date_sub(p.data_referencia, 6) AND p.data_referencia
  GROUP BY 1, 2
), recl AS (
  SELECT p.cliente_id, p.data_referencia, COUNT(*) AS reclamacoes_ultimos_30_dias
  FROM populacao_ref p
  JOIN {FQN}.reclamacoes_limpo c
    ON c.cliente_id = p.cliente_id
   AND to_date(c.data_reclamacao) BETWEEN date_sub(p.data_referencia, 29) AND p.data_referencia
  GROUP BY 1, 2
), pgt AS (
  SELECT p.cliente_id, p.data_referencia, MAX(to_date(g.data_pagamento)) AS ultimo_confirmado
  FROM populacao_ref p
  JOIN {FQN}.pagamentos g
    ON g.cliente_id = p.cliente_id
   AND g.status = 'confirmado'
   AND to_date(g.data_pagamento) <= p.data_referencia
  GROUP BY 1, 2
)
SELECT p.cliente_id, p.data_referencia,
       COALESCE(s.sessoes_ultimos_7_dias, 0) AS sessoes_ultimos_7_dias,
       COALESCE(r.reclamacoes_ultimos_30_dias, 0) AS reclamacoes_ultimos_30_dias,
       CASE WHEN g.ultimo_confirmado IS NULL THEN NULL
            ELSE datediff(p.data_referencia, g.ultimo_confirmado) END AS dias_desde_ultimo_pagamento
FROM populacao_ref p
LEFT JOIN sess s ON s.cliente_id = p.cliente_id AND s.data_referencia = p.data_referencia
LEFT JOIN recl r ON r.cliente_id = p.cliente_id AND r.data_referencia = p.data_referencia
LEFT JOIN pgt g ON g.cliente_id = p.cliente_id AND g.data_referencia = p.data_referencia""")

df_feat.createOrReplaceTempView("features_view")
print(f"Linhas de (cliente, data_referencia): {df_feat.count():,}")

In [0]:
# === 3.2 Gate de validação: os três clientes devem bater com os slides ==========
val = (df_feat.filter("cliente_id IN (101, 102, 103) AND data_referencia = '2026-06-01'")
             .select("cliente_id", "sessoes_ultimos_7_dias", "reclamacoes_ultimos_30_dias", "dias_desde_ultimo_pagamento")
             .orderBy("cliente_id")).toPandas()
esperado = {101: (1, 3, 24), 102: (18, 0, 2), 103: (2, 2, 22)}
ok = True
for _, r in val.iterrows():
    exp = esperado[r.cliente_id]
    got = (int(r.sessoes_ultimos_7_dias), int(r.reclamacoes_ultimos_30_dias),
           int(r.dias_desde_ultimo_pagamento))
    ok &= (exp == got)
    print(f"{'✔' if exp == got else '✘'} cliente {int(r.cliente_id)}: "
          f"sessões={got[0]}, reclamações={got[1]}, dias sem pagamento={got[2]} (esperado {exp})")
assert ok, "Features não correspondem ao slide! Verificar janelas de cálculo."

# Dados brutos -> feature calculada (Ana, data_referencia 2026-06-01)
ana_bruto = spark.sql(f"""
  SELECT to_date(data_evento) AS data, 'sessao' AS registro FROM {FQN}.eventos_uso_limpo
   WHERE cliente_id = 101 AND to_date(data_evento) BETWEEN '2026-05-26' AND '2026-06-01'
  UNION ALL
  SELECT to_date(data_reclamacao), concat('reclamação (', categoria, ')') FROM {FQN}.reclamacoes_limpo
   WHERE cliente_id = 101 AND to_date(data_reclamacao) BETWEEN '2026-05-03' AND '2026-06-01'
  UNION ALL
  SELECT to_date(data_pagamento), concat('pagamento ', status) FROM {FQN}.pagamentos
   WHERE cliente_id = 101 AND to_date(data_pagamento) <= '2026-06-01'
  ORDER BY data""").toPandas()
display(ana_bruto)
display(val)

In [0]:
# Instala o cliente de Databricks Feature Engineering (não vem pré-instalado
# no serverless). Leva ~30s; a próxima célula tem fallback caso falhe.
%pip install --quiet databricks-feature-engineering

In [0]:
# === 3.4 Persistir a feature table no Unity Catalog ==========================
# Preferido: cliente de Databricks Feature Engineering (feature table real
# no UC: chaves primárias + coluna timeseries). Fallback: Delta com PRIMARY KEY.
MODO_FEATURE_TABLE = None
try:
    from databricks.feature_engineering import FeatureEngineeringClient
    fe = FeatureEngineeringClient()
    spark.sql(f"DROP TABLE IF EXISTS {FQN}.features_churn")
    fe.create_table(
        name=f"{FQN}.features_churn",
        primary_keys=["cliente_id", "data_referencia"],
        timeseries_column="data_referencia",
        df=df_feat,
        description="Features de churn por (cliente, data_referencia) — corretas no ponto no tempo",
    )
    MODO_FEATURE_TABLE = "Feature Engineering client (UC feature table)"
except Exception as e:
    print(f"Cliente indisponível ({str(e)[:90]}...) — fallback: Delta + PRIMARY KEY.")
    spark.sql(f"""
      CREATE OR REPLACE TABLE {FQN}.features_churn (
        cliente_id BIGINT NOT NULL,
        data_referencia DATE NOT NULL,
        sessoes_ultimos_7_dias INT,
        reclamacoes_ultimos_30_dias INT,
        dias_desde_ultimo_pagamento INT,
        PRIMARY KEY (cliente_id, data_referencia TIMESERIES)
      )""")
    df_feat.write.mode("overwrite").insertInto(f"{FQN}.features_churn")
    MODO_FEATURE_TABLE = "Delta com PRIMARY KEY (fallback)"

for col, txt in [
    ("cliente_id", "Identificador único do cliente (chave primária)"),
    ("data_referencia", "Data da previsão; coluna timeseries da feature table"),
    ("sessoes_ultimos_7_dias", "Sessões entre data_referencia-6 e data_referencia (inclusivo)"),
    ("reclamacoes_ultimos_30_dias", "Reclamações entre data_referencia-29 e data_referencia (inclusivo)"),
    ("dias_desde_ultimo_pagamento", "Dias desde o último pagamento CONFIRMADO; NULL se nunca pagou"),
]:
    spark.sql(f"COMMENT ON COLUMN {FQN}.features_churn.{col} IS '{txt}'")
print("Modo:", MODO_FEATURE_TABLE, "| linhas:", scalar(f"SELECT COUNT(*) FROM {FQN}.features_churn"))

In [0]:
# === 3.5 Dicionário de features (a documentação mora JUNTO com os dados) ==============
dicionario = pd.DataFrame([
    {"Nome": "sessoes_ultimos_7_dias",
     "Significado": "Engajamento recente com o produto",
     "Fonte": f"{FQN}.eventos_uso_limpo",
     "Regra": "Sessões entre data_referencia−6 e data_referencia (inclusivo)",
     "Atualização planejada": "Diariamente, às 07:00",
     "Responsável": "Time de Dados"},
    {"Nome": "reclamacoes_ultimos_30_dias",
     "Significado": "Insatisfação recente",
     "Fonte": f"{FQN}.reclamacoes_limpo",
     "Regra": "Reclamações entre data_referencia−29 e data_referencia (inclusivo)",
     "Atualização planejada": "Diariamente, às 07:00",
     "Responsável": "Time de Dados"},
    {"Nome": "dias_desde_ultimo_pagamento",
     "Significado": "Saúde financeira / risco de inadimplência",
     "Fonte": f"{FQN}.pagamentos",
     "Regra": "datediff(data_referencia, último pagamento com status confirmado)",
     "Atualização planejada": "Diariamente, às 07:00",
     "Responsável": "Time de Dados"},
])
display(dicionario)

### Governança e linhagem no Unity Catalog

> Talk-track: "A feature table não é só uma tabela: chaves primárias, timeseries, comentários em português, controle de acesso e linhagem automática."

* **Controle de acesso**: `GRANT SELECT ON TABLE workspace.recife_dug.features_churn TO time_retencao` — o time vê as features, não os dados brutos de pagamentos.
* **Linhagem** (visível em *Catalog > tabela > Lineage*):
  `workspace.recife_dug.eventos_uso` → `workspace.recife_dug.eventos_uso_limpo` → `workspace.recife_dug.features_churn`
  `workspace.recife_dug.reclamacoes` → `workspace.recife_dug.reclamacoes_limpo` → `workspace.recife_dug.features_churn`
  `workspace.recife_dug.pagamentos` → `workspace.recife_dug.features_churn`
* **Chaves declaradas**: `PRIMARY KEY (cliente_id, data_referencia TIMESERIES)` — o consumidor sabe como unir sem ler nossa documentação.

## 4. Evitar vazamento de dados — o erro mais caro de ML

> Talk-track: "A 4ª reclamação da Ana chegou em 10/06 — NOVE dias depois da data da previsão. Usá-la é olhar o futuro, e o modelo pareceria mágico na avaliação e falharia em produção."

O modelo responde: **qual cliente cancela nos próximos 30 dias**, vendo **apenas** o que existia em 01/06.

In [0]:
# === 4.1 Feature correta vs ERRADA para o mesmo cliente ======================
# CORRETA (ponto no tempo): a janela termina na data_referencia (2026-06-01)
certo = scalar(f"""
  SELECT COUNT(*) FROM {FQN}.reclamacoes_limpo
  WHERE cliente_id = 101
    AND to_date(data_reclamacao) BETWEEN date_sub(to_date('2026-06-01'), 29)
                                     AND to_date('2026-06-01')""")
# ERRADA (vazamento): sem limite superior -> a reclamação futura (2026-06-10) entra
errado = scalar(f"""
  SELECT COUNT(*) FROM {FQN}.reclamacoes_limpo
  WHERE cliente_id = 101
    AND to_date(data_reclamacao) >= date_sub(to_date('2026-06-01'), 29)""")

comparacao = pd.DataFrame({
    "versão": ["Correta (point-in-time: data <= 01/06)", "Errada (sem corte no tempo)"],
    "filtro aplicado": ["data ENTRE 03/05 e 01/06", "data >= 03/05, sem limite superior"],
    "reclamacoes_ultimos_30_dias (Ana)": [certo, errado],
    "usa a reclamação de 10/06?": ["Não ✔", "Sim ✘ — vazamento de dados"],
})
display(comparacao)
assert (certo, errado) == (3, 4), f"Esperado correto=3 e errado=4; obtido {certo}/{errado}"
print("✔ Confirmado: só a versão ponto no tempo mantém o valor do slide (3).")

In [0]:
# === 4.2 Linha do tempo: o que o modelo pode e não pode ver ========================
import datetime as dt
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(12, 4))
marcos = [
    (dt.date(2026, 5, 26), "sessão (26/05)\njanela 7 dias", VERDE),
    (dt.date(2026, 5, 31), "sessão (31/05)", VERDE),
    (dt.date(2026, 6, 1), "01/06\nDATA DA PREVISÃO", "#ffffff"),
    (dt.date(2026, 6, 10), "10/06 reclamação\n(dado futuro, NÃO usar)", VERMELHO),
    (dt.date(2026, 6, 18), "18/06\ncancelamento", AMARELO),
    (dt.date(2026, 7, 1), "01/07 fim da janela\nde 30 dias (rótulo)", AZUL),
]
for i, (d, rotulo, cor) in enumerate(marcos):
    ax.plot([d], [0], "o", ms=11, color=cor, zorder=3)
    ax.annotate(rotulo, (d, 0), xytext=(d, 0.22 + (i % 2) * 0.38),
                ha="center", fontsize=9.5, color=cor)
# janela das features (verde) e janela do rótulo (amarelo)
ax.axvspan(dt.date(2026, 5, 26), dt.date(2026, 6, 1), color=VERDE, alpha=0.15)
ax.axvspan(dt.date(2026, 6, 2), dt.date(2026, 7, 1), color=AMARELO, alpha=0.10)
ax.annotate("janela das features (apenas dados até 01/06)", (dt.date(2026, 5, 29), -0.38),
            ha="center", fontsize=9, color=VERDE)
ax.annotate("janela do rótulo: cancela até 01/07?", (dt.date(2026, 6, 16), -0.60),
            ha="center", fontsize=9, color=AMARELO)
ax.set_xlim(dt.date(2026, 5, 21), dt.date(2026, 7, 7))
ax.set_ylim(-0.85, 1.05)
ax.set_yticks([])
ax.set_title("Linha do tempo da Ana — o modelo pode ver só o que está à esquerda de 01/06", pad=12)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
plt.tight_layout()
plt.show()

## 5. Rótulo + separação temporal — treinar no passado, avaliar no futuro

> Talk-track: "O rótulo é simples: cancelou nos 30 dias seguintes à data de referência? E separar treino/avaliação por TEMPO, nunca aleatório — em produção o modelo só vê o futuro."

* **TREINO**: 05/01 a 27/04 · **AVALIAÇÃO**: 04/05 a 25/05 · **SCORING** (novas previsões): 01/06

In [0]:
# === 5.1 Rótulo + separação temporal (nunca aleatória) =============================
df_ds = spark.sql(f"""
  SELECT f.cliente_id, f.data_referencia,
         f.sessoes_ultimos_7_dias, f.reclamacoes_ultimos_30_dias, f.dias_desde_ultimo_pagamento,
         c.nome,
         CASE WHEN a.data_cancelamento IS NOT NULL
              AND to_date(a.data_cancelamento) > f.data_referencia
              AND datediff(to_date(a.data_cancelamento), f.data_referencia) <= 30
              THEN 1 ELSE 0 END AS churn
  FROM {FQN}.features_churn f
  JOIN {FQN}.assinaturas a ON a.cliente_id = f.cliente_id
  LEFT JOIN {FQN}.clientes c ON c.cliente_id = f.cliente_id
""")
df_ds.createOrReplaceTempView("dataset_view")

def resumo_split(nome, ini, fim):
    d = df_ds.filter(f"data_referencia BETWEEN '{ini}' AND '{fim}'")
    n = d.count()
    pos = d.filter("churn = 1").count()
    taxa = pos / n if n else 0
    print(f"{nome:10s} {ini} → {fim}: {n:>7,} linhas | churn: {pos:>6,} ({taxa:.1%})")
    return {"split": nome, "data_referencia": f"{ini} → {fim}", "linhas": n, "churn": pos, "taxa_churn": round(taxa, 4)}

splits = pd.DataFrame([
    resumo_split("TREINO", "2026-01-05", "2026-04-27"),
    resumo_split("AVALIAÇÃO", "2026-05-04", "2026-05-25"),
    resumo_split("SCORING", "2026-06-01", "2026-06-01"),
])
display(splits)

In [0]:
# === 5.2 A separação na linha do tempo =================================================
fig, ax = plt.subplots(figsize=(11.5, 2.8))
faixas = [
    ("TREINO (05/01 → 27/04)", dt.date(2026, 1, 5), dt.date(2026, 4, 27), VERDE, 1),
    ("AVALIAÇÃO (04/05 → 25/05)", dt.date(2026, 5, 4), dt.date(2026, 5, 25), AZUL, 0),
    ("SCORING (01/06)", dt.date(2026, 6, 1), dt.date(2026, 6, 1), AMARELO, 1),
]
for nome, ini, fim, cor, desloca in faixas:
    ax.barh(desloca, max((fim - ini).days, 2), left=ini, height=0.5, color=cor, alpha=0.9)
    ax.annotate(nome, (ini + (fim - ini) / 2, desloca), ha="center", va="center",
                fontsize=10, fontweight="bold", color="#111318")
ax.set_yticks([])
ax.set_xlim(dt.date(2025, 12, 28), dt.date(2026, 6, 15))
ax.set_ylim(-0.6, 1.6)
ax.set_title("Separação TEMPORAL: o modelo treina no passado e é avaliado no futuro", pad=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
plt.tight_layout()
plt.show()
print("Nunca sorteamos clientes aleatoriamente entre treino e avaliação: datas fazem a separação.")

## 6. Modelagem com MLflow — treinamento é uma tentativa

> Talk-track: "Nenhum modelo nasce pronto. Cada treinamento é uma tentativa registrada: parâmetros, métricas, tags e o próprio modelo — tudo versionado no MLflow para comparar depois."

Três tentativas: **A** baseline Regressão Logística · **B** mesma com hiperparâmetros · **C** outro algoritmo (GradientBoosting).

In [0]:
# === 6.1 Conjuntos de treino/avaliação + experimento MLflow ==================================
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from mlflow.models import infer_signature

FEATURES = ["sessoes_ultimos_7_dias", "reclamacoes_ultimos_30_dias", "dias_desde_ultimo_pagamento"]

def conjunto(ini, fim):
    return (df_ds.filter(f"data_referencia BETWEEN '{ini}' AND '{fim}'")
                .select(*FEATURES, "churn").toPandas())

treino = conjunto("2026-01-05", "2026-04-27")
avaliacao = conjunto("2026-05-04", "2026-05-25")
X_treino, y_treino = treino[FEATURES], treino["churn"]
X_aval, y_aval = avaliacao[FEATURES], avaliacao["churn"]
print(f"TREINO: {len(treino):,} linhas ({y_treino.mean():.1%} churn) | "
      f"AVALIAÇÃO: {len(avaliacao):,} linhas ({y_aval.mean():.1%} churn)")

# caminho absoluto do experimento, na pasta do usuário no workspace
mlflow.set_experiment(EXPERIMENTO)
print("Experimento:", EXPERIMENTO)

In [0]:
# === 6.2 TRÊS treinamentos rastreados: baseline / tunado / outro algoritmo =============
def treinar_e_logar(nome_run, estimador, tag_hiperparam):
    # imputação + escala moram DENTRO do pipeline -> sem desvio entre treino e serving
    modelo = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("estimador", estimador),
    ])
    modelo.fit(X_treino, y_treino)
    proba = modelo.predict_proba(X_aval)[:, 1]
    pred = (proba >= 0.5).astype(int)
    metricas = {
        "precision": precision_score(y_aval, pred),
        "recall": recall_score(y_aval, pred),
        "f1": f1_score(y_aval, pred),
        "auc": roc_auc_score(y_aval, proba),
    }
    with mlflow.start_run(run_name=nome_run) as run:
        mlflow.log_params({"algoritmo": type(estimador).__name__, "hiperparametros": tag_hiperparam, "semente": SEED})
        mlflow.log_metrics(metricas)
        mlflow.set_tags({"etapa": "treinamento", "demo": "churn_recife_dug",
                         "algoritmo": type(estimador).__name__})
        # pyfunc_predict_fn="predict_proba": o endpoint devolve PROBABILIDADE (não a classe 0/1)
        assinatura = infer_signature(X_treino.head(20), modelo.predict_proba(X_treino.head(20)))
        mlflow.sklearn.log_model(modelo, name="model", signature=assinatura,
                                 input_example=X_treino.head(3), pyfunc_predict_fn="predict_proba")
        run_id = run.info.run_id
    print(f"{nome_run}: recall={metricas['recall']:.3f} precision={metricas['precision']:.3f} "
          f"f1={metricas['f1']:.3f} auc={metricas['auc']:.3f}")
    return {"tentativa": nome_run, "run_id": run_id, "modelo": modelo, **metricas}

run_A = treinar_e_logar("Treinamento A — Regressão Logística (baseline)",
                        LogisticRegression(random_state=SEED), "default")
run_B = treinar_e_logar("Treinamento B — Regressão Logística (tunada)",
                        LogisticRegression(C=0.3, class_weight="balanced",
                                          max_iter=1000, random_state=SEED),
                        "C=0.3, class_weight=balanced")
run_C = treinar_e_logar("Treinamento C — GradientBoosting",
                        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                    max_depth=3, random_state=SEED),
                        "n_estimators=200, lr=0.05, max_depth=3")

TENTATIVAS = [run_A, run_B, run_C]
display(pd.DataFrame([{k: round(v, 4) if isinstance(v, float) else v
                       for k, v in t.items() if k != "modelo"} for t in TENTATIVAS]))

In [0]:
# === 6.3 Escolher a vencedora PARA A DECISÃO =====================================
# Regra: recall primeiro (perder quem vai cancelar é caro), a precisão
# precisa ficar >= 0.50 para não inundar o time de retenção com falsos alertas.
elegiveis = [t for t in TENTATIVAS if t["precision"] >= 0.50]
if not elegiveis:
    print("Nenhuma tentativa com precisão >= 0.50; escolhendo maior F1.")
    MELHOR = max(TENTATIVAS, key=lambda t: t["f1"])
else:
    MELHOR = max(elegiveis, key=lambda t: (t["recall"], t["f1"]))

print(f"Escolhida: {MELHOR['tentativa']}")
print(f"recall = {MELHOR['recall']:.2%} | precision = {MELHOR['precision']:.2%} | "
      f"f1 = {MELHOR['f1']:.2%} | auc = {MELHOR['auc']:.2%}")
print("Critério: máximo recall com precisão >= 50% — capacidade do time de retenção.")

## 7. Avaliar para a decisão — não só métricas

> Talk-track: "A matriz de confusão só faz sentido na linguagem do negócio: quantos clientes sinalizamos, quantos acertamos, quantos falsos alertas — e quantos cancelamos sem ver."

In [0]:
# === 7.1 Limiar + matriz de confusão no estilo do slide (números reais) ==============
proba_aval = MELHOR["modelo"].predict_proba(X_aval)[:, 1]

# limiar: menor probabilidade com precisão >= 0.50 -> recall máximo que a
# capacidade do time de retenção suporta; fallback: maior F1 (mostrado com transparência)
candidatos_limiar = []
for t in np.round(np.arange(0.05, 0.96, 0.05), 2):
    pred_t = (proba_aval >= t).astype(int)
    candidatos_limiar.append((t,
                              precision_score(y_aval, pred_t, zero_division=0),
                              recall_score(y_aval, pred_t, zero_division=0),
                              f1_score(y_aval, pred_t, zero_division=0)))
tabela_limiares = pd.DataFrame(candidatos_limiar, columns=["limiar", "precisao", "recall", "f1"])
eleg = [c for c in candidatos_limiar if c[1] >= 0.50]
if eleg:
    escolhido = min(eleg, key=lambda c: c[0])
    modo = "menor limiar com precisão >= 50% → recall máximo"
else:
    escolhido = max(candidatos_limiar, key=lambda c: c[3])
    modo = "nenhum limiar atinge 50% de precisão — escolhido por maior F1"
LIMIAR = float(escolhido[0])
print(f"Limiar escolhido: {LIMIAR:.2f} ({modo})")
display(tabela_limiares.round(3))

pred = (proba_aval >= LIMIAR).astype(int)
X_sinalizados = int(pred.sum())
Y_acertos = int(((pred == 1) & (y_aval == 1)).sum())
Z_falsos = X_sinalizados - Y_acertos
W_perdidos = int(((pred == 0) & (y_aval == 1)).sum())
precisao = Y_acertos / X_sinalizados if X_sinalizados else 0
recall = Y_acertos / (Y_acertos + W_perdidos) if (Y_acertos + W_perdidos) else 0

print(f"{X_sinalizados} clientes sinalizados -> {Y_acertos} cancelaram (acerto), "
      f"{Z_falsos} continuaram (falso alerta); {W_perdidos} cancelaram sem ser sinalizados")
print(f"Precisão: {Y_acertos} de {X_sinalizados} = {precisao:.0%} / "
      f"Recall: {Y_acertos} de {Y_acertos + W_perdidos} = {recall:.0%}")
display(pd.DataFrame({
    "": ["Sinalizados", "Acertos (cancelaram)", "Falsos alertas (continuaram)",
         "Perdidos (cancelaram sem sinal)"],
    "clientes": [X_sinalizados, Y_acertos, Z_falsos, W_perdidos],
}))

### O trade-off Precisão × Recall na prática

> Talk-track: "Não existe limiar certo no abstrato — existe o limiar que a capacidade do time suporta."

* **Recall alto** = poucos churners perdidos, mas MUITOS contatos — se o time só liga para 100 clientes por semana, precisão baixa desperdiça essas ligações.
* **Precisão alta** = cada contato vale muito, mas churners passam direto sem ser vistos.
* A escolha aqui: **menor limiar que mantém precisão >= 50%** — no mínimo 1 em cada 2 clientes contatados realmente estava em risco. Ajuste ao longo do tempo conforme o time mede a taxa de sucesso dos contatos.

## 8. Registrar no Unity Catalog + testar e promover

> Talk-track: "O modelo escolhido ganha uma versão registrada no Unity Catalog — mas registrar NÃO é publicar: primeiro validamos contra critérios explícitos e, só se passar, ele vira champion."

Critérios: **recall >= 0.70** e **precisão >= 0.50** no conjunto de avaliação, e não piorar o champion atual (se existir).

In [0]:
# === 8.1 Registrar a vencedora + alias "candidate" ==============================
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient(registry_uri="databricks-uc")

model_uri = f"runs:/{MELHOR['run_id']}/model"
versao = mlflow.register_model(model_uri=model_uri, name=MODEL_UC, await_registration_for=300)
client.set_registered_model_alias(MODEL_UC, "candidate", versao.version)

alias_map = {}
for alvo in ("candidate", "champion"):
    try:
        alias_map[str(client.get_model_version_by_alias(MODEL_UC, alvo).version)] = alvo
    except Exception:
        pass
tabela_versoes = []
for v in client.search_model_versions(f"name='{MODEL_UC}'"):
    aliases = alias_map.get(str(v.version), "—")
    tabela_versoes.append({"versao": v.version, "aliases": aliases, "criado_em": str(v.creation_timestamp)})
display(pd.DataFrame(tabela_versoes).sort_values("versao", key=lambda s: s.astype(int), ascending=False))
print(f"Registrado: {MODEL_UC} v{versao.version} — alias 'candidate'")

In [0]:
# === 8.2 Gate de validação automatizada antes de promover ==========================
modelo_cand = mlflow.sklearn.load_model(f"models:/{MODEL_UC}@candidate")

# (a) checagem de assinatura: o modelo declara exatamente as nossas 3 features
pf = mlflow.pyfunc.load_model(f"models:/{MODEL_UC}@candidate")
nomes_entrada = (pf.metadata.signature.inputs.input_names()
                 if hasattr(pf.metadata.signature.inputs, "input_names")
                 else [c.name for c in pf.metadata.signature.inputs])
assert pf.metadata.signature is not None, "Modelo sem assinatura!"
assert set(nomes_entrada) == set(FEATURES), f"Assinatura divergente: {nomes_entrada}"
print("✔ Assinatura ok:", nomes_entrada)

# (b) avaliar o candidato no conjunto de avaliação, no mesmo limiar
proba_cand = modelo_cand.predict_proba(X_aval)[:, 1]
pred_cand = (proba_cand >= LIMIAR).astype(int)
prec_cand = precision_score(y_aval, pred_cand)
rec_cand = recall_score(y_aval, pred_cand)
print(f"Candidato: recall={rec_cand:.2%} | precisão={prec_cand:.2%} (limiar {LIMIAR:.2f})")

# (c) critérios explícitos + comparação com o champion atual (se existir)
criterios_ok = (rec_cand >= 0.70) and (prec_cand >= 0.50)
try:
    champ = client.get_model_version_by_alias(MODEL_UC, "champion")
    modelo_champ = mlflow.sklearn.load_model(f"models:/{MODEL_UC}/{champ.version}")
    pred_champ = (modelo_champ.predict_proba(X_aval)[:, 1] >= LIMIAR).astype(int)
    rec_champ = recall_score(y_aval, pred_champ)
    print(f"Champion atual (v{champ.version}): recall = {rec_champ:.2%}")
    passa_champion = rec_cand >= rec_champ
except Exception:
    rec_champ, passa_champion = None, True
    print("Ainda não existe champion — primeiro registro.")

if criterios_ok and passa_champion:
    client.set_registered_model_alias(MODEL_UC, "champion", versao.version)
    print(f"✔ Candidato APROVADO nos critérios e promovido a 'champion' (v{versao.version})")
else:
    print("Investigar o resultado e manter a versão atual")
display(pd.DataFrame({
    "critério": ["recall >= 0.70", "precisão >= 0.50", "recall >= champion"],
    "obtido": [f"{rec_cand:.2%}", f"{prec_cand:.2%}",
               f"{rec_cand:.2%} vs {rec_champ:.2%}" if rec_champ is not None else "sem champion"],
    "resultado": ["✔" if rec_cand >= 0.70 else "✘", "✔" if prec_cand >= 0.50 else "✘",
                   "✔" if passa_champion else "✘"],
}))

### Registrar uma versão NÃO é publicá-la

> Talk-track: "A versão entra no registro como candidata. Só depois de passar nos critérios — e não piorar o champion — ela é promovida. O time de negócio continua no controle."

* `candidate` → versão recém-registrada, aguardando validação
* `champion` → versão que produção consome (batch e serving leem o alias, nunca um número fixo de versão)
* Se a validação falha: **"Investigar o resultado e manter a versão atual"** — nada quebra em produção.

## 9. Batch scoring — a previsão vira uma tabela

> Talk-track: "Todo dia às 07:00, o job calcula as features do dia, aplica o champion e grava a lista de quem o time de retenção deve contatar. A produção consome o ALIAS, nunca um número de versão."

In [0]:
# === 9.1 Score em batch com o CHAMPION (alias, nunca um número de versão) =====
modelo_prod = mlflow.sklearn.load_model(f"models:/{MODEL_UC}@champion")

novo = (df_ds.filter("data_referencia = '2026-06-01'")
            .select("cliente_id", "nome", *FEATURES)).toPandas()
novo["probabilidade_churn"] = modelo_prod.predict_proba(novo[FEATURES])[:, 1]
# Ação em 3 faixas, ancoradas no limiar de produção (mesma régua para toda a base):
#   >= LIMIAR      -> risco Elevado -> Priorizar contato
#   >= LIMIAR / 2  -> risco Médio   -> Revisar antes de contatar
#   <  LIMIAR / 2  -> risco Baixo   -> Manter acompanhamento
novo["risco_estimado"] = np.select(
    [novo["probabilidade_churn"] >= LIMIAR, novo["probabilidade_churn"] >= LIMIAR / 2],
    ["Elevado", "Médio"], default="Baixo")
novo["acao_recomendada"] = np.select(
    [novo["probabilidade_churn"] >= LIMIAR, novo["probabilidade_churn"] >= LIMIAR / 2],
    ["Priorizar contato", "Revisar antes de contatar"], default="Manter acompanhamento")
novo["data_previsao"] = pd.Timestamp("2026-06-01")
novo["cliente_id"] = novo["cliente_id"].astype("int64")

(spark.createDataFrame(novo[["cliente_id", "nome", *FEATURES,
                             "probabilidade_churn", "acao_recomendada", "data_previsao"]])
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{FQN}.previsoes_churn"))

n_elevados = int((novo["risco_estimado"] == "Elevado").sum())
n_medios = int((novo["risco_estimado"] == "Médio").sum())
print(f"Previsões salvas em {FQN}.previsoes_churn: {len(novo):,} clientes | "
      f"{n_elevados} priorizados (risco >= {LIMIAR:.2f}) | {n_medios} para revisar "
      f"(risco >= {LIMIAR / 2:.2f})")

trio = novo[novo.cliente_id.isin([101, 102, 103])].set_index("cliente_id")
p_ana, p_bruno, p_carlos = (float(trio.loc[i, "probabilidade_churn"]) for i in (101, 102, 103))
print(f"\nAna: risco {p_ana:.1%} — {p_ana / max(p_bruno, 1e-9):.0f}x o risco de Bruno; "
      f"contato prioritário para risco >= {LIMIAR:.0%}, revisão para >= {LIMIAR / 2:.0%}")
print("Os três clientes do demo, como o dashboard mostraria:")
display(novo[novo.cliente_id.isin([101, 102, 103])]
        .sort_values("cliente_id")
        [["cliente_id", "nome", *FEATURES, "probabilidade_churn", "risco_estimado", "acao_recomendada"]])

## 10. Serving em tempo real — quando a previsão precisa ser instantânea

> Talk-track: "O batch cobre a operação semanal. Mas se o atendente quiser ver o risco NA HORA, durante o atendimento, publicamos o champion num endpoint REST."

A célula 10.1 resolve o alias `champion` no momento da execução e fixa no endpoint a versão correspondente — após uma promoção, basta re-executá-la para repontar o endpoint.

In [0]:
# === 10.1 Serving em tempo real: cria/atualiza o endpoint do champion ========
nome_endpoint = "churn-recife-dug"
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (EndpointCoreConfigInput, ServedEntityInput,
                                            ServingModelWorkloadType)

try:
    MODEL_UC, client
except NameError:
    MODEL_UC = "workspace.recife_dug.modelo_churn"
    from mlflow import MlflowClient
    client = MlflowClient(registry_uri="databricks-uc")

w = WorkspaceClient()

# resolve o alias 'champion' -> versão EXATA (a API de serving fixa a versão;
# re-executar esta célula após uma promoção reponta o endpoint para o novo champion)
versao_champion = client.get_model_version_by_alias(MODEL_UC, "champion").version

# CPU/Small é suficiente: modelo sklearn tabular com 3 features e artefato pequeno
entidade = ServedEntityInput(
    entity_name=MODEL_UC,
    entity_version=str(versao_champion),
    workload_type=ServingModelWorkloadType.CPU,
    workload_size="Small",
    scale_to_zero_enabled=True,   # sem custo parado; a 1ª requisição tem cold start
)

try:
    existente = w.serving_endpoints.get(nome_endpoint)
except NotFound:
    existente = None

try:
    if existente is None:
        w.serving_endpoints.create(
            name=nome_endpoint,
            config=EndpointCoreConfigInput(name=nome_endpoint, served_entities=[entidade]))
        print(f"✔ Criação aceita: '{nome_endpoint}' servirá {MODEL_UC} v{versao_champion} "
              f"(CPU, Small, scale-to-zero).")
        print("  A implantação é ASSÍNCRONA (~10–15 min) — rode a célula 10.3 para validar.")
    else:
        w.serving_endpoints.update_config(name=nome_endpoint, served_entities=[entidade])
        print(f"✔ Endpoint '{nome_endpoint}' já existia — config atualizada para "
              f"{MODEL_UC} v{versao_champion} (atualização assíncrona).")
except Exception as e:
    print(f"✘ Criação não concluída neste workspace: {type(e).__name__}: {str(e)[:200]}")
    print("Em produção, a criação equivaleria a:")
    display({"endpoint": nome_endpoint,
             "served_entity": f"{MODEL_UC} v{versao_champion}",
             "workload": "CPU / Small, scale-to-zero ativado",
             "inference_table": "captura de requisições para auditoria (opcional)"})

### Consumo em tempo real (formato da requisição)

```json
POST /serving-endpoints/churn-recife-dug/invocations
{"dataframe_records": [{"sessoes_ultimos_7_dias": 1,
                        "reclamacoes_ultimos_30_dias": 3,
                        "dias_desde_ultimo_pagamento": 24}]}
```

A resposta traz as probabilidades das duas classes; **a chance de churn é o segundo valor** — para a Ana, ~31,6%:

```json
{"predictions": [[0.684, 0.316]]}
```

> Talk-track: "As mesmas 3 features do treino, agora numa chamada REST. A resposta é a probabilidade — na hora, no painel do atendente."

In [0]:
# === 10.3 Validação do serving: payload, assinatura e resposta ================
import mlflow
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient

try:
    MODEL_UC, nome_endpoint
except NameError:
    MODEL_UC = "workspace.recife_dug.modelo_churn"
    nome_endpoint = "churn-recife-dug"
w = WorkspaceClient()

# perfil da Ana em 01/06 — o mesmo payload da célula 10.2
payload = {"sessoes_ultimos_7_dias": 1,
           "reclamacoes_ultimos_30_dias": 3,
           "dias_desde_ultimo_pagamento": 24.0}

# (a) o payload respeita a assinatura do modelo? (contrato que o endpoint exige)
pf = mlflow.pyfunc.load_model(f"models:/{MODEL_UC}@champion")
campos = (pf.metadata.signature.inputs.input_names()
          if hasattr(pf.metadata.signature.inputs, "input_names")
          else [c.name for c in pf.metadata.signature.inputs])
assert set(payload) == set(campos), f"Payload divergente da assinatura: {campos}"
print("✔ Payload compatível com a assinatura do champion:", campos)

# (b) resposta de referência: o que o endpoint DEVE devolver para este payload
saida_ref = np.asarray(pf.predict(pd.DataFrame([payload]))).ravel()
p_ref = float(saida_ref[-1])   # probabilidade da classe positiva (churn)
print(f"✔ Resposta de referência (local, champion): {p_ref:.1%} de churn para a Ana "
      f"— é isso que o endpoint deve devolver")

# (c) consulta REAL ao endpoint, se ele já estiver pronto
try:
    ep = w.serving_endpoints.get(nome_endpoint)
    ready = getattr(ep.state.ready, "value", ep.state.ready)
    if ready == "READY":
        resp = w.serving_endpoints.query(name=nome_endpoint, dataframe_records=[payload])
        saida = np.asarray(resp.predictions).ravel()
        p_endpoint = float(saida[-1])   # probabilidade da classe positiva (churn)
        dif = abs(p_endpoint - p_ref)
        print(f"✔ Endpoint '{nome_endpoint}' respondeu: {p_endpoint:.1%} "
              f"| diferença vs. referência local: {dif:.4f}")
        assert dif < 0.01, "Resposta do endpoint diverge da previsão local!"
    else:
        print(f"⏳ Endpoint em implantação (ready={ready}) — aguarde ~10–15 min "
              f"e re-execute esta célula para a validação ponta a ponta.")
except Exception as e:
    print(f"Endpoint indisponível para consulta ({type(e).__name__}: {str(e)[:120]}).")
    print("As validações (a)/(b) locais já garantem payload e resposta esperados.")

## 11. Monitoramento contínuo — modelo não é um produto estático

> Talk-track: "O comportamento dos clientes muda; a distribuição das features também. Se treino e produção divergirem, o modelo degrada em silêncio."

Teste de drift: comparamos a distribuição de cada feature no TREINO com a do dia de scoring (Kolmogorov–Smirnov).

In [0]:
# === 11.1 Drift de features: distribuição do treino vs. dia de scoring ==============
try:
    from scipy.stats import ks_2samp
    def p_valor(base, atual):
        return ks_2samp(base, atual).pvalue
except ImportError:
    def p_valor(base, atual):     # fallback: heurística de deslocamento da média
        pooled = np.std(np.concatenate([base, atual]))
        z = abs(np.mean(base) - np.mean(atual)) / (pooled / 10 or 1)
        return 0.0 if z > 2 else 1.0

tabela_drift = []
for f in FEATURES:
    base = treino[f].dropna().values
    atual = novo[f].dropna().values
    p = float(p_valor(base, atual))
    tabela_drift.append({"feature": f, "p_valor": round(p, 4),
                         "drift?": "SIM — investigar" if p < 0.05 else "não"})
display(pd.DataFrame(tabela_drift))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(treino["sessoes_ultimos_7_dias"].dropna(), bins=range(0, 40),
        density=True, alpha=.6, color=VERDE, label="treino")
ax.hist(novo["sessoes_ultimos_7_dias"].dropna(), bins=range(0, 40),
        density=True, alpha=.6, color=AZUL, label="scoring 01/06")
ax.set_xlabel("sessoes_ultimos_7_dias")
ax.set_ylabel("densidade")
ax.set_title("Distribuição da feature no treino vs. no dia de scoring")
ax.legend()
plt.tight_layout()
plt.show()

### No fluxo diário

* O teste de drift roda junto com o job das 07:00; p-valor < 0.05 em qualquer feature dispara investigação (e potencial re-treino).
* As métricas de classificação são recalculadas quando os cancelamentos reais dos 30 dias seguintes fecham — o rótulo se completa com o tempo.
* Um sinal prático de saúde: a **taxa de sinalizados por dia**. Um salto repentino costuma ser bug upstream (pipeline de dados), não mudança de comportamento.

## 12. Resumo — o que o time de dados entregou

De uma pergunta de negócio ("quem vai cancelar nos próximos 30 dias?") a um pipeline completo, com governança ponta a ponta:

| etapa | objeto criado |
| --- | --- |
| dados brutos | 5 tabelas em `workspace.recife_dug` com comentários |
| limpeza | `eventos_uso_limpo`, `reclamacoes_limpo` |
| features | `features_churn` (UC feature table, PK + timeseries) |
| experimentos | MLflow com tentativas A/B/C comparáveis |
| modelo | `workspace.recife_dug.modelo_churn` + aliases `candidate`/`champion` |
| produção | `previsoes_churn` (batch) e endpoint (opcional) |
| monitoramento | teste de drift + métricas contínuas |

In [0]:
# === 12.1 Checklist final — verificado ao vivo contra o catálogo ==============
tabelas = set(spark.sql(f"SHOW TABLES IN {FQN}").toPandas()["tableName"])
esperadas = ["assinaturas", "clientes", "eventos_uso", "reclamacoes", "pagamentos",
             "eventos_uso_limpo", "reclamacoes_limpo", "features_churn", "previsoes_churn"]
display(pd.DataFrame({
    "objeto": [f"{FQN}.{t}" for t in esperadas],
    "criado?": ["✔" if t in tabelas else "✘" for t in esperadas],
}))

exp = mlflow.get_experiment_by_name(EXPERIMENTO)
n_runs = len(mlflow.search_runs(experiment_ids=[exp.experiment_id]))
print(f"Experimento MLflow: {EXPERIMENTO} ({n_runs} tentativas registradas)")
print(f"Modelo registrado: {MODEL_UC} — aliases 'candidate' e 'champion'")
print(f"Limiar de produção: {LIMIAR:.2f} | features: {len(FEATURES)}")

### Próximos passos (fora do escopo do notebook)

* **Job diário 07:00 (America/Fortaleza)**: recalcula features → score com o champion → grava `previsoes_churn` → roda o teste de drift. Config equivalente (não criado neste workspace — fica como referência):

```json
{
  "name": "churn-recife-dug-score-diario",
  "schedule": {"quartz_cron_expression": "0 0 7 * * ?",
               "timezone_id": "America/Fortaleza"},
  "tasks": [{"task_key": "score_diario",
              "notebook_task": {"notebook_path": "/Users/lucas.almeida8@gmail.com/demo_churn_ml",
                                "base_parameters": {"secoes": "3.1,9,11"}}}],
  "max_concurrent_runs": 1
}
```
* **Permissões**: `GRANT SELECT` nas tabelas para o time de retenção — eles veem as previsões, não os dados brutos de pagamento.
* **Retreino**: quando o drift disparar ou as métricas caírem, o fluxo das seções 6–8 roda de novo — e a validação decide a promoção.

---
*Notebook de demonstração — dados sintéticos, nomes fictícios. Ana, Bruno e Carlos não existem (mas cancelam mesmo assim).*